In [28]:
import os

import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.multioutput import MultiOutputRegressor

load_dotenv()

True

In [29]:
# 讀取資料

path = os.environ.get("data_path")

data = pd.read_csv(path)

print('data size: ', data.shape)
print('columns: ', data.columns)
data.head(10)

data size:  (470, 24)
columns:  Index(['編號', '專案', '日期', '月', '日', '星期', '是否假日', '時長', '機位數量', '工作性質', '花絮',
       '視訊切換', '視訊連線', 'PA音控', '大場分小場', '導播人數', '攝影人數', '音控人數', '直播人數', '機動人數',
       '花絮人數', '視訊切換人數', '視訊連線人數', '人數'],
      dtype='object')


,編號,專案,日期,月,日,星期,是否假日,時長,機位數量,工作性質,...,大場分小場,導播人數,攝影人數,音控人數,直播人數,機動人數,花絮人數,視訊切換人數,視訊連線人數,人數
0,1,博思網路進場,2024/1/2,1,2,2,0,4.5,5,進場,...,0,1,4,1,1,1,0,0,0,8
1,2,博思網路,2024/1/3,1,3,3,0,5.0,5,直播,...,0,1,4,1,1,1,0,0,0,8
2,3,奕樂科技,2024/1/10,1,10,3,0,6.0,3,直播,...,0,1,2,1,1,0,0,0,0,5
3,4,DTxAWS(進),2024/1/10,1,10,3,0,4.0,4,進場,...,1,0,0,0,0,4,0,0,0,4
4,5,DTxAWS,2024/1/11,1,11,4,0,10.0,4,直播,...,1,1,3,5,1,0,0,0,0,10
5,6,GOOGLE,2024/1/12,1,12,5,0,3.0,3,錄製,...,0,0,3,1,0,1,0,0,0,5
6,7,高星x亞東醫院,2024/1/12,1,12,5,0,6.0,1,錄製,...,0,0,1,1,0,0,0,0,0,2
7,8,OPPO (進),2024/1/15,1,15,1,0,4.0,3,進場,...,0,1,2,0,2,0,0,0,2,7
8,9,OPPO,2024/1/16,1,16,2,0,7.0,3,直播,...,0,1,2,0,2,2,0,0,0,7
9,10,藝力國際x明基佳世達（進）,2024/1/15,1,15,1,0,3.5,0,進場,...,0,0,0,0,0,1,0,1,1,3


In [30]:
# 去除無關欄位，並將工作性質轉換成one-hot encode

data.drop(columns = ['編號', '專案', '日期'], inplace = True)
data = pd.get_dummies(data, columns=['工作性質'])

data.columns

Index(['月', '日', '星期', '是否假日', '時長', '機位數量', '花絮', '視訊切換', '視訊連線', 'PA音控',
       '大場分小場', '導播人數', '攝影人數', '音控人數', '直播人數', '機動人數', '花絮人數', '視訊切換人數',
       '視訊連線人數', '人數', '工作性質_直播', '工作性質_進場', '工作性質_錄製'],
      dtype='object')

In [40]:
# 將特徵與標籤分離成 x, y

X = data.drop(columns = ['導播人數', '攝影人數', '音控人數', '直播人數', '機動人數', '花絮人數', '視訊切換人數', '視訊連線人數', '人數'])
y = data.drop(columns = ['月', '日', '星期', '是否假日', '時長', '機位數量', '花絮', '視訊切換', '視訊連線', 'PA音控', '大場分小場', '工作性質_直播', '工作性質_進場', '工作性質_錄製'])

print(X.columns)
print('------------------')
print(y.columns)

Index(['月', '日', '星期', '是否假日', '時長', '機位數量', '花絮', '視訊切換', '視訊連線', 'PA音控',
       '大場分小場', '工作性質_直播', '工作性質_進場', '工作性質_錄製'],
      dtype='object')
------------------
Index(['導播人數', '攝影人數', '音控人數', '直播人數', '機動人數', '花絮人數', '視訊切換人數', '視訊連線人數',
       '人數'],
      dtype='object')


In [ ]:
# 將部分資料切割作為測試資料集

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

print('train size: ', X_train.shape)
print('test size: ', X_test.shape)

train size:  (376, 14)
test size:  (94, 14)


In [33]:
# 定義模型及參數網格

rf = RandomForestRegressor(random_state=42)
multi_rf = MultiOutputRegressor(rf)

parma_grid = {
    "estimator__n_estimators": [100, 130, 150, 170, 200, 230, 250],
    "estimator__max_depth": [None, 2, 5, 8, 10, 12, 15],
    "estimator__min_samples_split": [2, 5, 7, 10],
    "estimator__max_features": ['sqrt', 'log2']
}

In [34]:
# 執行網格搜尋

grid_search = GridSearchCV(multi_rf, param_grid=parma_grid,
                           cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)
print("網格搜尋執行完畢")
print("最佳參數組合：")
print(grid_search.best_params_)

Fitting 5 folds for each of 392 candidates, totalling 1960 fits
網格搜尋執行完畢
最佳參數組合：
{'estimator__max_depth': None, 'estimator__max_features': 'sqrt', 'estimator__min_samples_split': 2, 'estimator__n_estimators': 250}


In [35]:
# 查看搜尋結果

results_df = pd.DataFrame(grid_search.cv_results_)
results_df.head()

# mask = (results_df["param_estimator__n_estimators"] >= 300)
# df_mask = results_df[mask]

# df_mask

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_estimator__max_depth,param_estimator__max_features,param_estimator__min_samples_split,param_estimator__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,2.033247,0.143811,0.159053,0.008232,None,sqrt,2,100,"{'estimator__max_depth': None, 'estimator__max...",-0.360808,-0.271588,-0.400649,-0.344358,-0.437711,-0.363023,0.056049,47
1,2.639583,0.401430,0.197428,0.008751,None,sqrt,2,130,"{'estimator__max_depth': None, 'estimator__max...",-0.360854,-0.269321,-0.401726,-0.345631,-0.436820,-0.362870,0.056623,45
2,2.964811,0.124164,0.211783,0.025026,None,sqrt,2,150,"{'estimator__max_depth': None, 'estimator__max...",-0.360820,-0.270572,-0.398072,-0.345366,-0.438139,-0.362594,0.056096,39
3,3.595829,0.280335,0.281992,0.021913,None,sqrt,2,170,"{'estimator__max_depth': None, 'estimator__max...",-0.361016,-0.268386,-0.400538,-0.343894,-0.437030,-0.362173,0.056933,33
4,4.022024,0.515546,0.280283,0.062854,None,sqrt,2,200,"{'estimator__max_depth': None, 'estimator__max...",-0.358246,-0.265573,-0.399451,-0.336235,-0.434970,-0.358895,0.057715,7


In [36]:
# 取得最佳結果

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

In [37]:
# 評估模型成果

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"均方誤差 (MSE): {mse:.4f}")
print(f"均方根誤差 (RMSE): {rmse:.4f}")
print(f"R-squared 分數: {r2:.4f}")

均方誤差 (MSE): 0.2620
均方根誤差 (RMSE): 0.5119
R-squared 分數: 0.5026


In [41]:
save_path = os.environ.get("model_path")

model_data = {
    'model': best_model,
    'features': X.columns.tolist() # 紀錄訓練時的特徵順序
}

joblib.dump(best_model, save_path, compress=3)
print("模型保存完畢！")

模型保存完畢！
